# preparation des donnees

In [3]:
import sys
sys.path.append("..")           

import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sn
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from src.transformers import IQRClipper

## Transformation et préparation des données


In [11]:
data = pd.read_csv("../data/orig/student_mental_health.csv")

cible = "Mental_Health_Risk"
quant_discretes = ["Age", "Year_of_Study"]
quant_continues = ["Sleep_Hours", "Study_Hours_Per_Week", "GPA",
                   "Social_Media_Hours", "Screen_Time_Hours",
                   "Financial_Stress", "Family_Support", "Attendance_Rate"]
qualitatives = ["Gender", "Physical_Activity", "Part_Time_Job",
                "Diet_Quality", "Previous_Mental_Health_History"]

ordreRisque = {"Low": 0, "Medium": 1, "High": 2}
labelsRisque = list(ordreRisque)

X = data.drop(columns=["Student_ID", cible])
y = data[cible].map(ordreRisque)                    

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train :", X_train.shape, " Test :", X_test.shape)

Train : (1600, 15)  Test : (400, 15)


## Traitement des valeurs manquantes

In [15]:
valeursManquantes = X_train.isna().sum()
valeursManquantes[valeursManquantes > 0].sort_values(ascending=False)

Financial_Stress      71
Diet_Quality          53
Sleep_Hours           49
Social_Media_Hours    49
Family_Support        40
Attendance_Rate       34
GPA                   27
dtype: int64

In [ ]:
colonnesIteratives = ["GPA", "Social_Media_Hours", "Screen_Time_Hours","Study_Hours_Per_Week", "Year_of_Study"]

colonnesMediane = ["Sleep_Hours", "Financial_Stress", "Family_Support","Attendance_Rate", "Age"]

imputation = ColumnTransformer(
    transformers=[
        ("iteratif", IterativeImputer(max_iter=10, random_state=42), colonnesIteratives),
        ("mediane", SimpleImputer(strategy="median"), colonnesMediane),
    ],remainder="passthrough",verbose_feature_names_out=False, 
)

**Justification :** IterativeImputer pour GPA et Social_Media_Hours (corrélées à d'autres variables, erreur réduite de 17 % et 33 % par rapport à la médiane). Médiane pour les autres variables numériques (pas de prédicteur utile, distributions symétriques). Valeur la plus fréquente pour Diet_Quality (variable catégorielle). Suppression des lignes écartée : environ 20 % des lignes ont au moins un manquant.

### 2.3 Détection et traitement des valeurs aberrantes (IQR)

In [ ]:
lignes = []
for col in quant_continues:
    s = X_train[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    masque = (s < inf) | (s > sup)
    lignes.append({"variable": col, "borneInf": round(inf, 2), "borneSup": round(sup, 2),
                   "nbAberrantes": int(masque.sum()), "pourcentage": round(masque.mean() * 100, 2)})

tableauAberrantes = pd.DataFrame(lignes)
tableauAberrantes

In [ ]:
pt.figure(figsize=(16, 8))
for i, col in enumerate(quant_continues, 1):
    pt.subplot(2, 4, i)
    sn.boxplot(y=X_train[col])
    pt.title(col)
pt.tight_layout()
pt.show()

**Décision :** méthode IQR (1,5 × IQR) appliquée aux 8 variables continues. Les valeurs hors bornes sont **plafonnées** (pas supprimées) : elles sont rares (environ 0,5 % par colonne) et plausibles, donc on garde les lignes. IQR ne suppose pas de loi normale, plus robuste que le Z-score. Les variables discrètes (Age, Year_of_Study) ne sont pas traitées : toutes leurs valeurs sont légitimes. Les bornes sont apprises sur le train, puis appliquées au test.

### 2.4 Encodage des variables catégorielles

In [ ]:
colonnesOrdinales = ["Physical_Activity", "Diet_Quality"]
colonnesNominales = [c for c in qualitatives if c not in colonnesOrdinales]

assert set(colonnesNominales + colonnesOrdinales) == set(qualitatives)

pipelineNominal = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Ordre explicite : du plus faible au plus fort
ordreOrdinal = [["Low", "Moderate", "High"],        # Physical_Activity
                ["Poor", "Average", "Good"]]        # Diet_Quality

pipelineOrdinal = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OrdinalEncoder(categories=ordreOrdinal,
                                handle_unknown="use_encoded_value", unknown_value=-1)),
])

**Justification :** encodage ordinal avec ordre explicite pour Physical_Activity et Diet_Quality (ordre naturel). One-Hot pour les variables nominales (Gender, Part_Time_Job, Previous_Mental_Health_History) afin de ne pas créer d'ordre artificiel. Cible encodée Low = 0, Medium = 1, High = 2.

### 2.5 Gestion du déséquilibre des classes

In [ ]:
repartition = y_train.value_counts().sort_index()
ratio = repartition.max() / repartition.min()

print(repartition.rename(index={v: k for k, v in ordreRisque.items()}).to_dict())
print(f"Ratio classe majoritaire / minoritaire : {ratio:.2f}")
print("SMOTE nécessaire" if ratio > 1.5 else "SMOTE non nécessaire")

**Décision :** pas de SMOTE. Les classes sont équilibrées dans le train (ratio proche de 1). Un rééchantillonnage ne servirait à rien et ajouterait des données synthétiques.

### standardisation des variables numériques

In [ ]:
pipelineContinues = Pipeline([
    ("clipping", IQRClipper(factor=1.5)),             
    ("standardisation", StandardScaler()),
])

transformation = ColumnTransformer(
    transformers=[
        ("continues", pipelineContinues, quant_continues),
        ("discretes", StandardScaler(), quant_discretes),
        ("nominal", pipelineNominal, colonnesNominales),
        ("ordinal", pipelineOrdinal, colonnesOrdinales),
    ],
    remainder="drop",
)

**Justification :** StandardScaler (moyenne 0, écart-type 1) pour mettre les variables sur la même échelle, important pour la régression logistique. L'écrêtage IQR est placé avant la standardisation.

### 2.7 Regroupement dans un Pipeline (évite le data leakage)

In [ ]:
pretraitement = Pipeline([
    ("imputation", imputation),
    ("transformation", transformation),
]).set_output(transform="pandas")     
XTrainPrepare = pretraitement.fit_transform(X_train)    # tout est appris sur le train
XTestPrepare = pretraitement.transform(X_test)          # transform seulement

print("Train :", XTrainPrepare.shape, "| manquants :", XTrainPrepare.isna().sum().sum())
print("Test  :", XTestPrepare.shape, "| manquants :", XTestPrepare.isna().sum().sum())
XTrainPrepare.head()

**Justification :** imputation, écrêtage, standardisation et encodage sont dans un Pipeline : chaque paramètre (médiane, régressions de l'imputer, bornes IQR, moyenne et écart-type) est appris sur le train uniquement, puis appliqué au test. Pas de data leakage.